# Timor-Leste: shared data preparation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/cases/timor-leste-preparation.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/cases/timor-leste-preparation.ipynb)

**Starter** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup

Run the setup and data cells before timing either model. The notebook installs missing Python packages and provides two different optimization engines, CBC and HiGHS. Package installation and data loading are **not** part of model-instantiation or solving times. For local Jupyter use, see the [environment guidance](https://github.com/gromicho/teaching/blob/main/docs/SETUP.md).


## Inspect the facility-location case
This is the shared case workbook, not an instructor solution. Follow the ABW or AABW assignment for the particular model and questions. Sheet names, identifiers, units and distance conventions are part of the data contract; do not rely on row positions without checking their meaning.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'openpyxl': 'openpyxl', 'pandas': 'pandas',
                     'matplotlib': 'matplotlib', 'pyomo': 'pyomo',
                     'highspy': 'highspy'}
ensure_packages(required_packages)


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Prefer the checked-in file locally; Colab downloads the same frozen edition.
data_path = next((p for p in [Path("data/cases/timor-leste.xlsx"), Path("timor-leste.xlsx")]
                 if p.is_file()), Path("timor-leste.xlsx"))
if not data_path.is_file():
    url = "https://raw.githubusercontent.com/gromicho/teaching/main/data/cases/timor-leste.xlsx"
    with urlopen(url, timeout=45) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != "31431b07df5b6797630b9ec5023f81aacd76d5bdab4e21cdd3d9fe4e129f021a":
        raise ValueError("Dataset checksum mismatch; do not use an unverified copy.")
    data_path.write_bytes(payload)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == "31431b07df5b6797630b9ec5023f81aacd76d5bdab4e21cdd3d9fe4e129f021a", "Unexpected local data version"


In [ ]:
import pandas as pd
data = pd.read_excel(data_path,sheet_name=None,index_col=0)
for name,frame in data.items():
    print(name,frame.shape)
    display(frame.head())


In [ ]:
distances = data['Distances']
homes = data['Homes']
locations = data['Potential locations']

assert distances.index.equals(homes.index), 'Household identifiers differ.'
assert distances.columns.equals(locations.index), 'Facility identifiers differ.'
assert {'lon', 'lat'} <= set(homes.columns)
assert {'lon', 'lat'} <= set(locations.columns)
print(f'{len(homes)} households and {len(locations)} candidate facilities')


## Before modelling
Identify household identifiers, candidate locations and the distance matrix. Check whether distance is measured along roads or as a straight line, and confirm the unit before using a service threshold. Inspect missing values and duplicate identifiers. Distinguish the number of covered households from the number of people they represent.

Then use the course's released questions to define the opening and coverage decisions, service threshold and facility limit. Do not guess these choices from a previous year's solved example.


## Solvers for the assignment

The earlier Drive starter suggested `appsi_cbc` and `appsi_highs`. This edition still provides the **CBC** and **HiGHS** engines. Its shared `make_solver` helper uses Pyomo's `cbcnl` interface for the packaged CBC binary, so use `make_solver(name)` with the names below rather than passing `"cbc"` directly to `pyo.SolverFactory`. The choice of interface does not supply or solve either assignment model. Compare solver times only after setup, and record the machine and software used.


In [ ]:
import pyomo.environ as pyo
from teaching_utils import install_coin_solvers, make_solver

# Provides the CBC binary in Colab, Binder and local Jupyter environments.
install_coin_solvers()
solvers = ('cbc', 'appsi_highs')  # Distinct CBC and HiGHS engines.
for name in solvers:
    available = make_solver(name).available(exception_flag=False)
    print(f'{name}: {"available" if available else "unavailable"}')
    if not available:
        raise RuntimeError(f'Required assignment solver {name} is unavailable.')


## Visualize the data and a future result

The plot below reproduces the **data overview** in the earlier Drive notebook. Once you have solved your models, call `ShowFacilityLocation` with `X` containing the identifiers of opened facility sites and `Z` containing the identifiers of covered households. The helper only draws the sets you supply; it does not compute an optimal solution.


In [ ]:
import matplotlib.pyplot as plt

def ShowFacilityLocation(xHomes, yHomes, xFacilities, yFacilities, X=None, Z=None):
    """Plot candidate sites and households, optionally highlighting selected IDs."""
    home_points = pd.DataFrame({'lon': xHomes, 'lat': yHomes})
    facility_points = pd.DataFrame({'lon': xFacilities, 'lat': yFacilities})
    selected_sites = [] if X is None else list(X)
    covered_homes = [] if Z is None else list(Z)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(home_points['lon'], home_points['lat'], s=8, c='lightgray',
               label='Households')
    ax.scatter(facility_points['lon'], facility_points['lat'], s=35,
               marker='s', facecolors='none', edgecolors='tab:blue',
               label='Candidate sites')
    if covered_homes:
        covered = home_points.loc[covered_homes]
        ax.scatter(covered['lon'], covered['lat'], s=12, c='tab:green',
                   label='Covered households')
    if selected_sites:
        opened = facility_points.loc[selected_sites]
        ax.scatter(opened['lon'], opened['lat'], s=55, marker='s',
                   c='gold', edgecolors='black', label='Opened sites')
    ax.set(xlabel='Longitude', ylabel='Latitude', title='Timor-Leste facility-location data')
    ax.legend()
    plt.show()


In [ ]:
ShowFacilityLocation(homes['lon'], homes['lat'],
                     locations['lon'], locations['lat'])


In [ ]:
# TODO: prepare your index sets, validate units, and formulate the assigned model.
